In [12]:
import pandas as pd
from google.cloud import bigquery
import db_dtypes
import os

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] ='D:/mythings/BigqueryAPI/core-waters-440019-c6-3e5d7103ad61.json'

In [36]:
# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "hacker_news" dataset
dataset_ref = client.dataset("hacker_news", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

In [37]:
# List all the tables in the "hacker_news" dataset
tables = list(client.list_tables(dataset))
# Print names of all tables in the dataset (there are four!)
for table in tables:  
    print(table.table_id)

full


In [15]:
# Construct a reference to the "full" table
table_ref = dataset_ref.table("full")

# API request - fetch the table
table = client.get_table(table_ref)

In [16]:
table.schema

[SchemaField('title', 'STRING', 'NULLABLE', None, 'Story title', (), None),
 SchemaField('url', 'STRING', 'NULLABLE', None, 'Story url', (), None),
 SchemaField('text', 'STRING', 'NULLABLE', None, 'Story or comment text', (), None),
 SchemaField('dead', 'BOOLEAN', 'NULLABLE', None, 'Is dead?', (), None),
 SchemaField('by', 'STRING', 'NULLABLE', None, "The username of the item's author.", (), None),
 SchemaField('score', 'INTEGER', 'NULLABLE', None, 'Story score', (), None),
 SchemaField('time', 'INTEGER', 'NULLABLE', None, 'Unix time', (), None),
 SchemaField('timestamp', 'TIMESTAMP', 'NULLABLE', None, 'Timestamp for the unix time', (), None),
 SchemaField('type', 'STRING', 'NULLABLE', None, 'type of details (comment comment_ranking poll story job pollopt)', (), None),
 SchemaField('id', 'INTEGER', 'NULLABLE', None, "The item's unique id.", (), None),
 SchemaField('parent', 'INTEGER', 'NULLABLE', None, 'Parent comment ID', (), None),
 SchemaField('descendants', 'INTEGER', 'NULLABLE', N

In [17]:
client.list_rows(table, max_results=5).to_dataframe()

,title,url,text,dead,by,score,time,timestamp,type,id,parent,descendants,ranking,deleted
0,None,None,None,<NA>,None,<NA>,1438497614,2015-08-02 06:40:14+00:00,story,9990003,<NA>,<NA>,<NA>,<NA>
1,None,None,None,<NA>,None,<NA>,1438497660,2015-08-02 06:41:00+00:00,story,9990004,<NA>,<NA>,<NA>,<NA>
2,None,None,None,<NA>,None,<NA>,1438501105,2015-08-02 07:38:25+00:00,story,9990077,<NA>,<NA>,<NA>,<NA>
3,None,None,None,<NA>,None,<NA>,<NA>,NaT,story,99902,<NA>,<NA>,<NA>,<NA>
4,None,None,None,<NA>,None,<NA>,1438507076,2015-08-02 09:17:56+00:00,story,9990213,<NA>,<NA>,<NA>,<NA>


In [22]:
# Preview the first five entries in the "by" column of the "full" table
client.list_rows(table, selected_fields=table.schema[:5], max_results=5).to_dataframe()

,title,url,text,dead,by
0,None,None,None,<NA>,None
1,None,None,None,<NA>,None
2,None,None,None,<NA>,None
3,None,None,None,<NA>,None
4,None,None,None,<NA>,None


In [23]:
dataset_ref = client.dataset("openaq", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)
tables = list(client.list_tables(dataset))
for table in tables:  
    print(table.table_id)

global_air_quality


In [24]:
table_ref = dataset_ref.table('global_air_quality')
table = client.get_table(table_ref)
client.list_rows(table, max_results= 5).to_dataframe()

,location,city,country,pollutant,value,timestamp,unit,source_name,latitude,longitude,averaged_over_in_hours,location_geom
0,"Borówiec, ul. Drapałka",Borówiec,PL,bc,0.85217,2022-04-28 07:00:00+00:00,µg/m³,GIOS,1.0,52.276794,17.074114,POINT(52.276794 1)
1,"Kraków, ul. Bulwarowa",Kraków,PL,bc,0.91284,2022-04-27 23:00:00+00:00,µg/m³,GIOS,1.0,50.069308,20.053492,POINT(50.069308 1)
2,"Płock, ul. Reja",Płock,PL,bc,1.41000,2022-03-30 04:00:00+00:00,µg/m³,GIOS,1.0,52.550938,19.709791,POINT(52.550938 1)
3,"Elbląg, ul. Bażyńskiego",Elbląg,PL,bc,0.33607,2022-05-03 13:00:00+00:00,µg/m³,GIOS,1.0,54.167847,19.410942,POINT(54.167847 1)
4,"Piastów, ul. Pułaskiego",Piastów,PL,bc,0.51000,2022-05-11 05:00:00+00:00,µg/m³,GIOS,1.0,52.191728,20.837489,POINT(52.191728 1)


In [32]:
query = """
        SELECT city
        FROM `bigquery-public-data.openaq.global_air_quality`
        WHERE country = 'US'
        """
client = bigquery.Client()
query_job = client.query(query)

In [33]:
us_cities = query_job.to_dataframe()

In [34]:
us_cities

,city
0,HOWARD
1,HOWARD
2,HOWARD
3,HOWARD
4,HOWARD
...,...
1421346,New York-Northern New Jersey-Long Island
1421347,New York-Northern New Jersey-Long Island
1421348,New York-Northern New Jersey-Long Island
1421349,New York-Northern New Jersey-Long Island


In [35]:
query = """ Select city, country from `bigquery-public-data.openaq.global_air_quality`
where country = 'US'
"""
query_job = client.query(query)
us_cities = query_job.to_dataframe()
us_cities

,city,country
0,HOWARD,US
1,HOWARD,US
2,HOWARD,US
3,HOWARD,US
4,HOWARD,US
...,...,...
1421346,New York-Northern New Jersey-Long Island,US
1421347,New York-Northern New Jersey-Long Island,US
1421348,New York-Northern New Jersey-Long Island,US
1421349,New York-Northern New Jersey-Long Island,US


In [38]:
# Query to get the score column from every row where the type column has value "job"
query = """
        SELECT score, title
        FROM `bigquery-public-data.hacker_news.full`
        WHERE type = "job" 
        """

# Create a QueryJobConfig object to estimate size of query without running it
dry_run_config = bigquery.QueryJobConfig(dry_run=True)

# API request - dry run query to estimate costs
dry_run_query_job = client.query(query, job_config=dry_run_config)

print("This query will process {} bytes.".format(dry_run_query_job.total_bytes_processed))

This query will process 665211879 bytes.


In [41]:
# Only run the query if it's less than 1 MB
ONE_MB = 1000*1000
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=ONE_MB)

# Set up the query (will only run if it's less than 1 MB)
safe_query_job = client.query(query, job_config=safe_config)

# API request - try to run the query, and return a pandas DataFrame
safe_query_job.to_dataframe()

,score,title
0,1,Kamcord is looking for iOS engineers who want ...
1,1,LE TOTE (YC S13) Is Hiring a Growth Engineer
2,1,Automatic (YC S11) Is Hiring a Principal Serve...
3,1,Join Shoptiques (YC W12) in Inside Sales
4,1,Flexport is hiring engineers who like ships an...
...,...,...
17332,4,Etacts is looking for Software Engineer #1 (an...
17333,5,Mixpanel (analytics) looking for engineer #1
17334,7,Spend the Summer with BackType and True Ventures
17335,16,[SF] Justin.tv: Hackers Wanted


In [ ]:
# Only run the query if it's less than 1 GB
ONE_GB = 1000*1000*1000
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=ONE_GB)

# Set up the query (will only run if it's less than 1 GB)
safe_query_job = client.query(query, job_config=safe_config)

# API request - try to run the query, and return a pandas DataFrame
job_post_scores = safe_query_job.to_dataframe()

# Print average score for job posts
job_post_scores.score.mean()

In [53]:
# Query to select prolific commenters and post counts
prolific_commenters_query = """
select `by` as author, count(1) as NumPosts
from `bigquery-public-data.hacker_news.full`
group by author
having count(1) > 10000
""" # Your code goes here

# Set up the query (cancel the query if it would use too much of 
# your quota, with the limit set to 1 GB)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
query_job = client.query(prolific_commenters_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
prolific_commenters = query_job.to_dataframe()

# View top few rows of results
print(prolific_commenters.head())

          author  NumPosts
0        mcguire     10081
1      gus_massa     11964
2            rdl     10361
3          gruez     14873
4  nickpsecurity     10237


In [47]:
df = client.list_rows(table, max_results= 100).to_dataframe()

In [59]:
query = '''
select count(deleted) from `bigquery-public-data.hacker_news.full`
where deleted is True
'''
client.query(query).to_dataframe()

,f0_
0,0


In [61]:
# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "world_bank_intl_education" dataset
dataset_ref = client.dataset("world_bank_intl_education", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "international_education" table
table_ref = dataset_ref.table("international_education")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the "international_education" table
client.list_rows(table, max_results=10).to_dataframe()

,country_name,country_code,indicator_name,indicator_code,value,year
0,Chad,TCD,"Enrolment in lower secondary education, both s...",UIS.E.2,321921.0,2012
1,Chad,TCD,"Enrolment in upper secondary education, both s...",UIS.E.3,68809.0,2006
2,Chad,TCD,"Enrolment in upper secondary education, both s...",UIS.E.3,30551.0,1999
3,Chad,TCD,"Enrolment in upper secondary education, both s...",UIS.E.3,79784.0,2007
4,Chad,TCD,"Repeaters in primary education, all grades, bo...",UIS.R.1,282699.0,2006
5,Chad,TCD,"Repeaters in primary education, all grades, bo...",UIS.R.1,169600.0,1991
6,Chad,TCD,"Repeaters in primary education, all grades, bo...",UIS.R.1,79342.0,1977
7,Chad,TCD,"Repeaters in primary education, all grades, bo...",UIS.R.1,251163.0,2001
8,Chad,TCD,"Repeaters in primary education, all grades, bo...",UIS.R.1,224397.0,2000
9,Chad,TCD,"Teachers in lower secondary education, both se...",UIS.T.2,2703.0,2000


In [68]:
# Your code goes here
country_spend_pct_query = """
                          SELECT country_name
                          FROM `bigquery-public-data.world_bank_intl_education.international_education`
                          WHERE indicator_code = 'SE.XPD.TOTL.GD.ZS' and (year between 2010 and 2018) 
                          """

# Set up the query (cancel the query if it would use too much of 
# your quota, with the limit set to 1 GB)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
country_spend_pct_query_job = client.query(country_spend_pct_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
country_spending_results = country_spend_pct_query_job.to_dataframe()
print(country_spending_results.head())

  country_name
0         Cuba
1         Mali
2         Mali
3         Mali
4         Oman
